# 10. Target Encoding y Retícula Decimal

Los notebooks 1 a 9 exprimieron modelos y combinaciones sobre **una sola representación de los
datos**: las columnas crudas más siete ratios. Todo lo que sacamos de ahí se movió en el cuarto
decimal — tuning +0,0003, stacking +0,0002.

Este notebook cambia la representación, y hay tres razones medidas para creer que ahí sí hay algo.

### 1. Las variables continuas tienen pocos valores distintos y muchas filas por valor

`daily_screen_time_hours` tiene **1.390 valores distintos sobre 691.369 filas**: unas **497 filas por
nivel**. Con esa densidad, la media del target por valor exacto es una cantidad **bien estimada** en
1.390 puntos. Un árbol aproxima esa misma curva con unas pocas decenas de cortes.

### 2. Dos columnas no son cantidades, son claves de una tabla de lookup

| columna | sd de la tasa por valor | sd esperada por muestreo | ratio | \|dif\| entre valores **adyacentes** |
|---|---|---|---|---|
| `app_opens_per_day` | 0,1952 | 0,0087 | **22,6x** | **0,2170** |
| `notifications_per_day` | 0,1919 | 0,0111 | **17,3x** | **0,2248** |

Valores enteros vecinos difieren en promedio **0,22** en tasa de adicción, con miles de filas detrás
de cada uno. Eso es 17-22 veces más de lo que el muestreo puede producir: **el generador usa el valor
exacto como clave**, y 87 no tiene relación con 88.

Esto explica un hallazgo que en los notebooks 3 y 4 dejamos mal atribuido. Ahí vimos que estas dos
columnas quedaban 4ª y 5ª en importancia pese a correlación lineal ~0, y lo adjudicamos a
"interacciones". El mecanismo real es otro: el árbol estaba **reconstruyendo la tabla de lookup a
fuerza de splits**. Un target encoding la lee directo.

### 3. Los dígitos decimales llevan señal que el valor no lleva

El primer decimal de `daily_screen_time_hours` mueve la tasa de adicción **0,0852** entre dígitos, con
más de 51.000 filas por dígito. Nada del comportamiento de una persona explica eso: es huella de cómo
el generador fabricó los números.

Y **el target encoding no puede verla**, porque estima cada valor exacto por separado; no tiene forma
de decir "todo lo que termina en ,2 comparte algo". Es un canal distinto, no otra vista del mismo.

> **Tiempo de ejecución: ~1,5 a 2 horas.** La comparación de las tres configuraciones —que es la parte
> informativa— termina alrededor de los 70 minutos; el modelo final es opcional y va al final.

**Métrica:** ROC AUC · **Dataset:** Playground Series S6E8

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import time
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

SEED, N_FOLDS, SMOOTH = 42, 5, 10.0
print(f"pandas {pd.__version__}")

## 1. Carga y Diagnóstico

In [ ]:
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")
y = train["addicted_label"].values
NTR, NTE = len(train), len(test)

NUMS = ["age", "daily_screen_time_hours", "weekend_screen_time", "social_media_hours",
        "gaming_hours", "work_study_hours", "sleep_hours", "notifications_per_day",
        "app_opens_per_day"]
CATS = ["gender", "stress_level", "academic_work_impact"]
ENC_COLS = NUMS + CATS

FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(NTR), y))

print(f"{'columna':<26}{'niveles':>9}{'filas/nivel':>13}")
for c in ENC_COLS:
    k = train[c].astype(object).fillna("__m__").astype(str).nunique()
    print(f"{c:<26}{k:>9,d}{NTR/k:>13,.0f}")
print("\n  Todas las columnas superan las ~480 filas por nivel: el target encoding es estimable.")

## 2. Los Niveles, y una Trampa de pandas

Cada columna se pasa a **nivel de texto**, con los faltantes como un nivel explícito. El idioma
obvio —`df[c].astype(str)`— es correcto en pandas 2.x, donde escribe la cadena `"nan"`, pero en
pandas ≥ 3.0 el nuevo dtype `str` **preserva el NA**: el `groupby` descarta silenciosamente cada fila
faltante de las estadísticas, el `.map()` devuelve NA, y el `fillna` posterior les entrega la tasa
base. Nada falla, nada avisa, y se pierde el encoding sobre el 4-20 % de cada columna.

`.astype(object).fillna("__missing__").astype(str)` es correcto en ambas versiones. Y en vez de
confiar, se verifica: **el `groupby` tiene que cubrir las 691.369 filas**.

In [ ]:
def levels(df):
    return pd.DataFrame({c: df[c].astype(object).fillna("__missing__").astype(str).values
                         for c in ENC_COLS})

LTR, LTE = levels(train), levels(test)

for c in ENC_COLS:
    cubiertas = pd.DataFrame({"lv": LTR[c].values, "y": y}).groupby("lv")["y"].size().sum()
    assert cubiertas == NTR, f"{c}: el groupby cubre {cubiertas:,} de {NTR:,} filas"
print(f"Cobertura verificada: las {NTR:,} filas entran en las estadisticas de las "
      f"{len(ENC_COLS)} columnas.")
print(f"Ejemplo — niveles de gender: {sorted(LTR['gender'].unique())}")

## 3. Target Encoding Anidado (y por qué el anidado no es opcional)

El encoding usa el target, así que una versión descuidada infla el CV y se derrumba en el
leaderboard. El esquema es de dos niveles:

- Para cada **fold externo**, las estadísticas salen únicamente de su porción de entrenamiento.
- Dentro de esa porción, cada fila recibe un encoding calculado con un **fold interno que la excluye**.

Así **ninguna fila ve nunca su propio target**, ni directa ni indirectamente. La media suavizada es
`(n·media_nivel + s·media_global) / (n + s)` con `s = 10`, que es lo que evita que un nivel con tres
filas arrastre al modelo.

**La frecuencia es distinta y va aparte.** Contar cuántas veces aparece un valor **no usa el target**,
así que se puede contar sobre `train + test` juntos: son 987.671 filas en vez de las ~553.000 de un
fold de entrenamiento, y es una estimación estrictamente mejor de qué tan común es un valor. Las
features de test se conocen al momento de predecir; sólo las etiquetas están ocultas.

In [ ]:
# --- Frecuencia: transductiva, una sola vez, sin tocar el target ---
FQ_TR, FQ_TE = {}, {}
for c in ENC_COLS:
    conteo = pd.concat([LTR[c], LTE[c]], ignore_index=True).value_counts()
    FQ_TR[f"fq_{c}"] = LTR[c].map(conteo).astype(np.float32).values
    FQ_TE[f"fq_{c}"] = LTE[c].map(conteo).astype(np.float32).values
FQ_TR, FQ_TE = pd.DataFrame(FQ_TR), pd.DataFrame(FQ_TE)
print(f"Frecuencias transductivas sobre {NTR + NTE:,} filas "
      f"(un fold de entrenamiento solo ve {len(FOLDS[0][0]):,}).")


def mapas_desde(lv, yy):
    """Estadisticas por nivel a partir de un subconjunto de filas."""
    gm, m = yy.mean(), {}
    for c in ENC_COLS:
        g = pd.DataFrame({"lv": lv[c].values, "y": yy}).groupby("lv")["y"].agg(["count", "mean"])
        m[c] = (((g["count"] * g["mean"] + SMOOTH * gm) / (g["count"] + SMOOTH)).astype(np.float32),
                gm)
    return m


def aplicar(mapas, lv):
    return pd.DataFrame({f"te_{c}": lv[c].map(mapas[c][0]).fillna(mapas[c][1]).astype(np.float32).values
                         for c in ENC_COLS})


def encoding_anidado_con(itr, iva, yy):
    """TE para el fold externo (itr, iva). Ninguna fila ve su propio target.

    Recibe el target como parametro para poder correr el test de fuga de mas abajo
    con las etiquetas permutadas.
    """
    e_tr = pd.DataFrame(np.zeros((len(itr), len(ENC_COLS)), np.float32),
                        columns=[f"te_{c}" for c in ENC_COLS])
    interno = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1)
    for i_fit, i_apl in interno.split(np.zeros(len(itr)), yy[itr]):
        mapas = mapas_desde(LTR.iloc[itr[i_fit]], yy[itr[i_fit]])
        e_tr.iloc[i_apl] = aplicar(mapas, LTR.iloc[itr[i_apl]]).values

    mapas_full = mapas_desde(LTR.iloc[itr], yy[itr])         # sin las filas de validacion
    return e_tr, aplicar(mapas_full, LTR.iloc[iva]), aplicar(mapas_full, LTE)


def encoding_anidado(itr, iva):
    return encoding_anidado_con(itr, iva, y)


t0 = time.time()
_e_tr, _e_va, _e_te = encoding_anidado(*FOLDS[0])
print(f"\nPrueba sobre el fold 1: {time.time()-t0:.0f}s por fold")
print(f"  formas: train {_e_tr.shape}, val {_e_va.shape}, test {_e_te.shape}")
print(f"  AUC del te_ de app_opens_per_day sobre validacion: "
      f"{roc_auc_score(y[FOLDS[0][1]], _e_va['te_app_opens_per_day']):.5f}")

### Verificación de fuga, antes de gastar dos horas

La prueba definitiva de que un encoding no filtra: **permutar el target**. Con las etiquetas
barajadas no queda ninguna relación real entre features y target, así que un encoding correctamente
anidado tiene que dar **AUC ≈ 0,5**. Si diera más, estaría filtrando información de la fila que
califica.

Se corre también la versión **ingenua** —estadísticas calculadas sobre todo el train, sin anidar— como
contraste, para ver qué aspecto tiene una fuga cuando la hay.

In [ ]:
rng = np.random.default_rng(0)
y_perm = rng.permutation(y)
FOLDS_P = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(NTR), y_perm))
P_RAPIDO = dict(objective="binary:logistic", eval_metric="auc", tree_method="hist",
                n_estimators=60, learning_rate=0.1, max_depth=5, random_state=SEED, n_jobs=-1)

t0 = time.time()
oof_p = np.zeros(NTR)
for itr, iva in FOLDS_P:
    e_tr, e_va, _ = encoding_anidado_con(itr, iva, y_perm)
    m = XGBClassifier(**P_RAPIDO).fit(e_tr, y_perm[itr])
    oof_p[iva] = m.predict_proba(e_va)[:, 1]
auc_perm = roc_auc_score(y_perm, oof_p)

# contraste: version ingenua, estadisticas de todo el train
mapas_todo = mapas_desde(LTR, y_perm)
e_todo = aplicar(mapas_todo, LTR)
oof_n = np.zeros(NTR)
for itr, iva in FOLDS_P:
    m = XGBClassifier(**P_RAPIDO).fit(e_todo.iloc[itr], y_perm[itr])
    oof_n[iva] = m.predict_proba(e_todo.iloc[iva])[:, 1]

print(f"{'='*66}")
print("TEST DE FUGA — target permutado, solo features te_")
print(f"{'='*66}")
print(f"  encoding ANIDADO (el que usa este notebook): {auc_perm:.5f}")
print(f"  encoding INGENUO (sin anidar), de contraste: {roc_auc_score(y_perm, oof_n):.5f}")
print(f"{'='*66}")
print(f"  ~0.5 = sin fuga.  Claramente por encima = fuga.   ({(time.time()-t0)/60:.0f} min)")
assert auc_perm < 0.53, f"POSIBLE FUGA: {auc_perm:.5f} con target permutado"
print("\n  OK: el esquema anidado no filtra.")

## 4. La Retícula Decimal

Dos features por columna con parte fraccionaria: la posición sub-unidad continua (`frac_`) y el primer
dígito decimal (`d1_`). Es información que el target encoding no puede representar, porque agrupa por
valor exacto y no por la forma del número.

In [ ]:
FRAC_COLS = ["daily_screen_time_hours", "weekend_screen_time", "social_media_hours",
             "gaming_hours", "work_study_hours", "sleep_hours"]


def reticula(df):
    o = {}
    for c in FRAC_COLS:
        v = df[c].values
        o[f"frac_{c}"] = v - np.floor(v)
        o[f"d1_{c}"]   = np.floor(v * 10) % 10
    return pd.DataFrame(o).astype(np.float32)


LAT_TR, LAT_TE = reticula(train), reticula(test)
print(f"Retícula: {LAT_TR.shape[1]} features sobre {len(FRAC_COLS)} columnas")

tabla = (pd.DataFrame({"d1": LAT_TR["d1_daily_screen_time_hours"], "y": y})
           .dropna().groupby("d1")["y"].agg(tasa="mean", filas="size"))
print(f"\nTasa de adiccion por primer decimal de daily_screen_time_hours:")
print(tabla.round(4).to_string())
print(f"\n  spread = {tabla['tasa'].max() - tabla['tasa'].min():.4f}  (tasa base {y.mean():.4f})")

## 5. Las Features Base

Las mismas de los notebooks 2 a 6, para que la comparación aísle exactamente el efecto del encoding.

In [ ]:
def agregar_derivadas(df):
    d = df.copy()
    componentes = ["social_media_hours", "gaming_hours", "work_study_hours"]
    d["otro_uso"]      = d["daily_screen_time_hours"] - d[componentes].sum(axis=1)
    d["ocio_horas"]    = d["social_media_hours"] + d["gaming_hours"]
    d["screen_sleep"]  = d["daily_screen_time_hours"] / d["sleep_hours"]
    d["social_ratio"]  = d["social_media_hours"] / d["daily_screen_time_hours"]
    d["gaming_ratio"]  = d["gaming_hours"] / d["daily_screen_time_hours"]
    d["weekend_ratio"] = d["weekend_screen_time"] / d["daily_screen_time_hours"]
    d["min_per_open"]  = d["daily_screen_time_hours"] * 60 / d["app_opens_per_day"]
    return d.replace([np.inf, -np.inf], np.nan)


derivadas = ["otro_uso", "ocio_horas", "screen_sleep", "social_ratio",
             "gaming_ratio", "weekend_ratio", "min_per_open"]
tr_fe, te_fe = agregar_derivadas(train), agregar_derivadas(test)
for col in CATS:
    niveles = sorted(set(tr_fe[col].dropna()) | set(te_fe[col].dropna()))
    dtype = pd.CategoricalDtype(categories=niveles, ordered=False)
    tr_fe[col] = tr_fe[col].astype(dtype)
    te_fe[col] = te_fe[col].astype(dtype)

BASE_COLS = NUMS + derivadas + CATS
X_base, X_base_test = tr_fe[BASE_COLS], te_fe[BASE_COLS]
print(f"Features base: {X_base.shape[1]}")

## 6. Validación — Tres Configuraciones sobre los Mismos Folds

Se usan los hiperparámetros que encontró Optuna en el notebook 6, pero con `learning_rate=0,1` en
lugar de 0,05: eso reduce a la mitad la cantidad de árboles y se aplica **igual a las tres
configuraciones**, así que la comparación es justa. Los AUC absolutos van a quedar algo por debajo de
los del notebook 6; lo que importa acá son las diferencias.

In [ ]:
PARAMS = dict(
    objective="binary:logistic", eval_metric="auc", tree_method="hist",
    enable_categorical=True, n_estimators=4000, learning_rate=0.1,
    max_depth=5, min_child_weight=13, subsample=0.8469, colsample_bytree=0.6757,
    reg_lambda=0.4489, reg_alpha=0.001045, gamma=0.5520,
    early_stopping_rounds=100, random_state=SEED, n_jobs=-1,
)


def evaluar(usar_enc, usar_lat, etiqueta):
    t0 = time.time()
    oof = np.zeros(NTR, np.float32); test_sum = np.zeros(NTE, np.float32); iters = []

    for itr, iva in FOLDS:
        Xa = X_base.iloc[itr].reset_index(drop=True)
        Xb = X_base.iloc[iva].reset_index(drop=True)
        Xt = X_base_test.reset_index(drop=True)

        if usar_enc:
            e_tr, e_va, e_te = encoding_anidado(itr, iva)
            Xa = pd.concat([Xa, e_tr.reset_index(drop=True),
                            FQ_TR.iloc[itr].reset_index(drop=True)], axis=1)
            Xb = pd.concat([Xb, e_va.reset_index(drop=True),
                            FQ_TR.iloc[iva].reset_index(drop=True)], axis=1)
            Xt = pd.concat([Xt, e_te.reset_index(drop=True), FQ_TE], axis=1)

        if usar_lat:
            Xa = pd.concat([Xa, LAT_TR.iloc[itr].reset_index(drop=True)], axis=1)
            Xb = pd.concat([Xb, LAT_TR.iloc[iva].reset_index(drop=True)], axis=1)
            Xt = pd.concat([Xt, LAT_TE], axis=1)

        m = XGBClassifier(**PARAMS)
        m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], verbose=False)
        oof[iva] = m.predict_proba(Xb)[:, 1]
        test_sum += m.predict_proba(Xt)[:, 1] / N_FOLDS
        iters.append(m.best_iteration)

    auc = roc_auc_score(y, oof)
    print(f"  {etiqueta:<38s} OOF {auc:.6f}  | {Xa.shape[1]:>3d} features "
          f"| {np.mean(iters):>5.0f} arboles | {(time.time()-t0)/60:.0f} min")
    return auc, oof, test_sum, m


print(f"{'='*92}")
print("COMPARACION — mismos folds, mismos hiperparametros")
print(f"{'='*92}")
a_base, oof_base, tst_base, _ = evaluar(False, False, "A. base (notebooks 2-6)")
a_enc,  oof_enc,  tst_enc,  _ = evaluar(True,  False, "B. + target y frequency encoding")
a_lat,  oof_lat,  tst_lat,  m_lat = evaluar(True, True, "C. + reticula decimal")
print(f"{'='*92}")
print(f"  encoding aporta   {a_enc - a_base:+.6f}")
print(f"  reticula aporta   {a_lat - a_enc:+.6f}")
print(f"  total             {a_lat - a_base:+.6f}")
print(f"{'='*92}")

## 7. ¿Es Real o Está Filtrando?

Una fuga de target no se ve como un CV malo: se ve como **CV y leaderboard divergiendo**. Por eso
conviene predecir el leaderboard **antes** de enviar.

Nuestro offset CV→LB medido en el episodio fue de +0,00156 (XGBoost) y +0,00222 (logit), y se achica
a medida que el modelo mejora. Con esa referencia, la configuración C debería aterrizar en torno a
`OOF + 0,0012`. Si el score público queda muy por debajo de esa predicción, hay fuga.

In [ ]:
print(f"OOF de la configuracion C : {a_lat:.6f}")
print(f"Prediccion del leaderboard: {a_lat + 0.0012:.5f}  (offset de ~+0.0012)")
print(f"  Referencia: XGBoost del notebook 3 dio 0.96478 OOF -> 0.96634 publico")
print(f"\n  Si el publico queda muy por debajo de la prediccion, el encoding esta filtrando.")

imp = pd.Series(m_lat.feature_importances_, index=m_lat.feature_names_in_).sort_values()
top = imp.tail(20)
fig, ax = plt.subplots(figsize=(11, 8))
colores = ["tomato" if n.startswith(("te_", "fq_")) else
           "seagreen" if n.startswith(("frac_", "d1_")) else "steelblue" for n in top.index]
ax.barh(top.index, top.values, color=colores, edgecolor="black")
ax.set_xlabel("Importancia (gain)")
ax.set_title("Top 20 — rojo: encoding · verde: reticula · azul: features base", fontsize=13)
plt.tight_layout()
plt.show()

for pref, nombre in [(("te_", "fq_"), "encoding"), (("frac_", "d1_"), "reticula")]:
    sel = imp[[n.startswith(pref) for n in imp.index]]
    print(f"  {nombre:9s}: {len(sel):>2d} features, {sel.sum()/imp.sum():.1%} de la importancia total")

## 8. Modelo Final y Submission

Sólo si la configuración C ganó. Se reentrena con `learning_rate=0,05` —el régimen de los notebooks 3
y 6— para que el resultado sea comparable con ellos y utilizable por el stack del notebook 9.

**Esta celda agrega ~40 minutos.** La comparación de la sección 6 ya respondió la pregunta; esto
produce el artefacto.

In [ ]:
PARAMS_FINAL = {**PARAMS, "learning_rate": 0.05, "n_estimators": 8000}

t0 = time.time()
oof_f = np.zeros(NTR, np.float32); tst_f = np.zeros(NTE, np.float32); iters = []
for itr, iva in FOLDS:
    e_tr, e_va, e_te = encoding_anidado(itr, iva)
    Xa = pd.concat([X_base.iloc[itr].reset_index(drop=True), e_tr.reset_index(drop=True),
                    FQ_TR.iloc[itr].reset_index(drop=True),
                    LAT_TR.iloc[itr].reset_index(drop=True)], axis=1)
    Xb = pd.concat([X_base.iloc[iva].reset_index(drop=True), e_va.reset_index(drop=True),
                    FQ_TR.iloc[iva].reset_index(drop=True),
                    LAT_TR.iloc[iva].reset_index(drop=True)], axis=1)
    Xt = pd.concat([X_base_test.reset_index(drop=True), e_te.reset_index(drop=True),
                    FQ_TE, LAT_TE], axis=1)
    m = XGBClassifier(**PARAMS_FINAL)
    m.fit(Xa, y[itr], eval_set=[(Xb, y[iva])], verbose=False)
    oof_f[iva] = m.predict_proba(Xb)[:, 1]
    tst_f += m.predict_proba(Xt)[:, 1] / N_FOLDS
    iters.append(m.best_iteration)

auc_f = roc_auc_score(y, oof_f)
print(f"Modelo final: OOF {auc_f:.6f} | {np.mean(iters):.0f} arboles | {(time.time()-t0)/60:.0f} min")
print(f"  vs XGBoost+Optuna del notebook 6: {auc_f - 0.965073:+.6f}")

np.save("../models/10_te_oof.npy",  oof_f)
np.save("../models/10_te_test.npy", tst_f)
submission = pd.DataFrame({"id": test["id"], "addicted_label": tst_f})
submission.to_csv("../submissions/10_te_submission.csv", index=False)

sample = pd.read_csv("../data/sample_submission.csv")
assert list(submission.columns) == list(sample.columns)
assert (submission["id"].values == sample["id"].values).all()
assert submission["addicted_label"].between(0, 1).all()
assert submission["addicted_label"].notna().all()
print("\nSubmission escrita y verificada: ../submissions/10_te_submission.csv")
print("  El inventario de 8_envio_final.ipynb la levanta sola.")

## 9. Conclusiones

_Pendiente de completar tras ejecutar el notebook._ Lo que conviene registrar:

- **Cuánto aportó el encoding** y **cuánto la retícula**, por separado. La referencia externa que
  motivó el notebook reportaba +0,0023 para el encoding y +0,00011 para la retícula.
- **Qué fracción de la importancia se llevan las features nuevas.** Si el encoding domina el top 20,
  confirma que el modelo estaba gastando splits en reconstruir a mano lo que ahora recibe hecho.
- **La comparación entre el leaderboard predicho y el real.** Es el chequeo de fuga, y el único que
  importa: un encoding que filtra da CV alto y público bajo.
- **Y lo más importante para el episodio**: si esto funciona, la conclusión de que "~0,966 era el
  techo práctico" era una afirmación sobre **nuestra representación**, no sobre el problema. Los
  notebooks 1 a 9 optimizaron modelos sobre features fijas; acá se cambió lo único que no habíamos
  tocado.